<font size = "4">

**Two-dimensional Array Operations**

- Two dimensional arrays (e.g. matrices) are still stored in memory as 1-D arrays.

- There are two major paradigms for storing 2D arrays: row-major ordering or column-major ordering.

- In Fortran, 2D arrays are stored in column-major fashion. In C, they are stored in row-major ordering.

- So this is sometimes called "C" ordering and "F" ordering (but "C" ordering is **not** column-major!!)

- Consider the following matrix:

$$\begin{bmatrix} 1 & 2 & 3\\
                    4 & 5 & 6\\
                    7 & 8 & 9\end{bmatrix}$$

- In row-major ("C") ordering, the entries are stored in a 1D array as follows:

$$\begin{bmatrix} 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9\end{bmatrix}$$

- In column-major ("F") ordering, the entries are stored in a 1D array as follows:

$$\begin{bmatrix} 1 & 4 & 7 & 2 & 5 & 8 & 3 & 6 & 9\end{bmatrix}$$

- When implementing a linear algebra computation, it can be advantageous to write it to take advantage of the ordering of the elements.

- **Python and NumPy uses row-major ("C") ordering**

<font size = "4">

**Forward Substitution**

- Recall the problem of solving a lower-triangular linear system:

$$
\underbrace{\begin{bmatrix}
\ell_{11} & 0         & \cdots & 0 \\
\ell_{21} & \ell_{22} & \ddots & \vdots \\
\vdots    & \vdots    & \ddots & 0 \\
\ell_{n1} & \ell_{n2} & \cdots & \ell_{nn}
\end{bmatrix}}_{L}
\underbrace{\begin{bmatrix}
x_1 \\ x_2 \\ \vdots \\ x_n
\end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix}
b_1 \\ b_2 \\ \vdots \\ b_n
\end{bmatrix}}_{\mathbf{b}}.
$$

We assume $\ell_{ii} \neq 0$ for every $i = 1, 2, \dots, n$ so that a unique solution exists.

The *forward-substitution* algorithm can be used to compute the solution as follows:

$$\begin{align*}
x_1 &= \frac{b_1}{\ell_{11}}\\[2pt]
x_2 &= \frac{b_2 - \ell_{21}x_1}{\ell_{22}}\\
x_3 &= \frac{b_3 - \ell_{31}x_1 - \ell_{32}x_2}{\ell_{33}}\\
&\vdots\\
x_i &= \frac{b_i - \sum_{j=1}^{i-1}\ell_{ij}x_j}{\ell_{ii}}\\
&\vdots\\
x_n &= \frac{b_n - \sum_{j=1}^{n-1}\ell_{ij}x_j}{\ell_{nn}}
\end{align*}$$

- Notice that we access the element of $L$ in row-wise fashion:


$$\ell_{11}, \ell_{21}, \ell_{22}, \ell_{31}, \ell_{32}, \ell_{33}, ...$$

- So the following `solve_triangular` function would be a good choice for C or Python:

In [1]:
import numpy as np

def solve_triangular(L, b):
    # C ordering (row-major)
    n = b.size
    x = np.zeros(n,) 
    for i in range(n):
        x[i] = b[i] 
        for j in range(i):
            x[i] -= L[i,j]*x[j]
        x[i] /= L[i,i]
    return x

<font size = "4">

- However, we could re-order the necessary computations for forward substitution as follows:

    1. Copy $x_i \leftarrow b_i$ for each $i$

    2. $x_1 \leftarrow \frac{x_1}{\ell_{11}}$

    3. $x_j \leftarrow x_j - \ell_{j1}x_1$ for $j > 1$

    4. $x_2 \leftarrow \frac{x_2}{\ell_{22}}$

    5. $x_j \leftarrow x_j - \ell_{j2}x_2$ for $j > 2$

    6. $x_3 \leftarrow \frac{x_3}{\ell_{33}}$

    7. $x_j \leftarrow x_j - \ell_{j3}x_3$ for $j > 3$

    8. ...and so on...

- This would access the elements of $L$ in the following order:

$$\ell_{11}, \ell_{21}, \ell_{31},\dots, \ell_{n1}, \ell_{22}, \ell_{23}, \ell_{24}, ...$$

- So the following `solve_triangular` function would be a good choice for Fortran or Matlab:

In [2]:
def solve_triangular_F(L, b):
    # Fortran ordering (column-major)
    n = b.size
    x = b.copy()
    for i in range(n):
        x[i] /= L[i,i]
        for j in range(i+1, n):
            x[j] -= L[j,i]*x[i]
        
    return x

<font size = "4">

Check that they both work...

In [3]:
n = 50

L = np.tril(np.random.randn(n,n))
L += np.diag(5*(np.random.rand(n,) + 1)) # ensure matrix is "sufficiently" non-singular

x_true = np.ones(n,)
b = L @ x_true 

my_x = solve_triangular(L, b)

# true solution should be all ones... should get a really small number here
print(max(abs(my_x - 1)))

my_x2 = solve_triangular_F(L, b)
print(max(abs(my_x2 - 1)))


1.1102230246251565e-15
1.1102230246251565e-15


<font size = "4">

What is the difference in timing?

In [4]:
from timeit import default_timer as timer

n = 3000
n_repeats = 12

c_time = 0
f_time = 0

L = np.tril(np.random.randn(n,n))
L += np.diag(5*(np.random.rand(n,) + 1))
b = np.random.randn(n,)

for _ in range(n_repeats):

    start = timer()
    solve_triangular(L,b)
    stop = timer()

    c_time += (stop - start)


for _ in range(n_repeats):

    start = timer()
    solve_triangular_F(L,b)
    stop = timer()

    f_time += (stop - start)


print("C ordering:", c_time / n_repeats)
print("Fortran ordering:", f_time / n_repeats)

print(f"C ordering is {f_time / c_time} times faster than F ordering")

C ordering: 2.2581957079166464
Fortran ordering: 2.5957587022499715
C ordering is 1.1494834983300681 times faster than F ordering


<font size = "4">

Is there really no difference?

In this case yes, because the cost is dominated by Python compilation overhead. To see the effect of access ordering we need to **vectorize** our code, allowing the inner for-loop to be handled in C using arrays:

In [8]:
import numpy as np

def solve_triangular_vec(L, b):
    n = b.size
    x = np.zeros(n,) 
    for i in range(n):
        x[i] = b[i] - np.dot(L[i,:i], x[:i])
        x[i] /= L[i,i]
    return x

def solve_triangular_F_vec(L, b):
    # Fortran ordering
    n = b.size
    x = b.copy()
    for i in range(n):
        x[i] /= L[i,i]
        x[i+1:] -= L[i+1:, i] * x[i] # more writes here (see below)
    return x

In [6]:
n = 50

L = np.tril(np.random.randn(n,n))
L += np.diag(5*(np.random.rand(n,) + 1)) # ensure matrix is "sufficiently" non-singular

x_true = np.ones(n,)
b = L @ x_true 

my_x = solve_triangular_vec(L, b)

# true solution should be all ones... should get a really small number here
print(max(abs(my_x - 1)))


my_x2 = solve_triangular_F_vec(L, b)
print(max(abs(my_x2 - 1)))

4.440892098500626e-16
1.1102230246251565e-15


In [9]:
from timeit import default_timer as timer

n = 3000
n_repeats = 12

c_time = 0
f_time = 0

L = np.tril(np.random.randn(n,n))
L += np.diag(5*(np.random.rand(n,) + 1))
b = np.random.randn(n,)

solve_triangular_vec(L,b)
for _ in range(n_repeats):

    start = timer()
    solve_triangular_vec(L,b)
    stop = timer()

    c_time += (stop - start)


solve_triangular_F_vec(L,b)
for _ in range(n_repeats):

    start = timer()
    solve_triangular_F_vec(L,b)
    stop = timer()

    f_time += (stop - start)




print("C ordering:", c_time / n_repeats)
print("Fortran ordering:", f_time / n_repeats)

print(f"C ordering is {f_time / c_time} times faster than F ordering")

C ordering: 0.00878104533332665
Fortran ordering: 0.03641015400002819
C ordering is 4.146448699204518 times faster than F ordering


<font size = "4">

- If you have an algorithm that uses column-order accesses, there is still hope.

- You can use `np.asfortranarray` to convert to the column-major storage.

In [10]:
from timeit import default_timer as timer

n = 3000
n_repeats = 12

c_time = 0
f_time = 0

L = np.tril(np.random.randn(n,n))
L += np.diag(5*(np.random.rand(n,) + 1))

# converting L to column-order here
L = np.asfortranarray(L)


b = np.random.randn(n,)



solve_triangular_vec(L,b)
for _ in range(n_repeats):

    start = timer()
    solve_triangular_vec(L,b)
    stop = timer()

    c_time += (stop - start)

solve_triangular_F_vec(L,b)
for _ in range(n_repeats):

    start = timer()
    solve_triangular_F_vec(L,b)
    stop = timer()

    f_time += (stop - start)




print("C ordering:", c_time / n_repeats)
print("Fortran ordering:", f_time / n_repeats)


print(f"F ordering is {c_time / f_time} times faster than C ordering")

C ordering: 0.028423212833293594
Fortran ordering: 0.013758047916629343
F ordering is 2.065933554348105 times faster than C ordering


<font size = "4">

- We only get a $\approx 1.5$ times speed-up here. This is because the column-based algorithm has to overwrite `x[i+1:]`. Writing data is more expensive than reading data.
